# Improving GRPO for Reinforcement Learning

In [10]:
import pathlib
import torch
import sympy
import tokenizers

In [3]:
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer

device = get_device()
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using CPU
✓ qwen3\qwen3-0.6B-base.pth already up-to-date


In [4]:
print(tokenizer.encode("<think>"))

[13708, 766, 29]


In [5]:
tokenizer._tok.add_special_tokens(["<tool_response>", "</tool_response>", "<think>", "</think>"])

4

In [6]:
print(tokenizer.encode("<think>"))
print(tokenizer.encode("</think>"))

[151667]
[151668]


Reward for "...think....</think(>"

In [7]:
def reward_format(token_ids, prompt_len,
                start_think_id=151667, end_think_id=151668):
    try:
        gen = token_ids[prompt_len:].tolist()
        return float(gen.index(start_think_id) < gen.index(end_think_id))
    except ValueError:
        return 0.0

In [8]:
def render_prompt_with_think_tokens(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "When solving the problem, first write your reasoning inside <think> and </think> tags.\n"
        "Then write the final result on a new line in the exact format:\n"
        "\\boxed{ANSWER}\n\n"
        f"Question:\n{prompt}\n\nAnswer:"
    )
    return template

In [11]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter

@torch.no_grad()
def sample_response(model, tokenizer, prompt, device, max_new_tokens=512, temperature = 0.8, top_p = 0.9):
    input_ids = torch.tensor(tokenizer.encode(prompt), device=device)
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache = cache)[:, -1]

    generated = []
    for i in range(max_new_tokens):
        if temperature != 0.0 or temperature is not None:
            logits = logits/temperature
        
        prob = torch.softmax(logits, dim = -1)
        prob = top_p_filter(prob, top_p)

        next_token = torch.multinomial(prob.cpu(), 1).to(device)
        token_id = next_token.item()
        generated.append(token_id)

        if(tokenizer.eos_token_id is not None and tokenizer.eos_token_id == token_id):
            break

        logits = model(next_token, cache = cache)[:, -1]

    full_token_ids = torch.cat([input_ids,torch.tensor(generated, device=device, dtype=input_ids.dtype)])
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [12]:
from reasoning_from_scratch.ch03 import (extract_final_candidate, grade_answer)
def reward_rlvr(answer_text, ground_truth):
    answer = extract_final_candidate(answer_text, fallback=None)
    if not answer:
        return 0.0
    correct = grade_answer(answer, ground_truth)
    return float(correct)        

In [13]:
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(1, token_ids[1:].unsqueeze(-1)).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:]) #sum not mean since this naturally penalizes longer answers

### GRPO + Thinking

In [ ]:
def compute_grpo_loss_with_think(model, tokenizer, example, device, num_rollouts,
                                max_new_tokens = 256, temperature = 0.8, top_p = 0.9, think_weight = 1):
    prompt = render_prompt_with_think_tokens(example['problem'])
    roll_logps, roll_rewards, samples = [], [], []
    assert num_rollouts >= 2

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        token_id, prompt_len, response = sample_response(model, tokenizer, prompt, device, max_new_tokens, temperature, top_p)
        rlvr_reward = reward_rlvr(response, example['answer'])
        
        give_format_reward = False
        if rlvr_reward:
            give_format_reward = True
        
        format_reward = reward_format(token_id, prompt_len)
        if(give_format_reward):
            reward = rlvr_reward + think_weight*format_reward
        
        roll_rewards.append(reward)
        
        logps = sequence_logprob(model, token_id, prompt_len)
        roll_logps.append(logps)
        samples.append({
                "text": response,
                "reward": reward,
                "gen_len": token_id.numel() - prompt_len,
            })
    
    if was_training:
        model.train()
    
    rewards = torch.tensor(roll_rewards, device=device)

    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-6) 

    logps = torch.stack(roll_logps)

    policy_gradient_loss = -(advantages.detach() * logps).mean()

    loss = policy_gradient_loss 

    return {
        "loss": loss.item(),
        "pg_loss": policy_gradient_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

### GRPO + KL + Thinking

In [22]:
import copy

kl_coeff = 0.2

#reference model initialisation
if kl_coeff:
    ref_model = copy.deepcopy(model).to(device)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad = False
else:
    ref_model = None

def compute_grpo_loss_with_kl_with_think(model, tokenizer, example, ref_model, device, num_rollouts, kl_coeff,
                                max_new_tokens = 256, temperature = 0.8, top_p = 0.9, think_weight = 1):
    prompt = render_prompt_with_think_tokens(example['problem'])
    roll_logps, roll_ref_logps, roll_rewards, samples = [], [], [], []
    assert num_rollouts >= 2

    was_training = model.training
    model.eval()


    for _ in range(num_rollouts):
        token_id, prompt_len, response = sample_response(model, tokenizer, prompt, device, max_new_tokens, temperature, top_p)
        rlvr_reward = reward_rlvr(response, example['answer'])
        
        give_format_reward = False
        if rlvr_reward:
            give_format_reward = True
        
        format_reward = reward_format(token_id, prompt_len)
        if(give_format_reward):
            reward = rlvr_reward + think_weight*format_reward
        
        roll_rewards.append(reward)
        
        #reference model
        if kl_coeff:
            with torch.no_grad():
                ref_logps = sequence_logprob(ref_model, token_id, prompt_len)
            roll_ref_logps.append(ref_logps)
        

        #training model
        logps = sequence_logprob(model, token_id, prompt_len)
        roll_logps.append(logps)


        samples.append({
                "text": response,
                "reward": reward,
                "gen_len": token_id.numel() - prompt_len,
            })
    
    if was_training:
        model.train()
    
    rewards = torch.tensor(roll_rewards, device=device)

    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-6) 

    logps = torch.stack(roll_logps)

    if kl_coeff:
        ref_logps = torch.stack(roll_ref_logps)

    policy_gradient_loss = -(advantages.detach() * logps).mean()

    kl_loss = torch.tensor(0.0, device=device)
    if kl_coeff:
        kl_loss = kl_coeff*torch.mean(logps-ref_logps)

    loss = policy_gradient_loss + kl_loss

    return {
        "loss": loss.item(),
        "pg_loss": policy_gradient_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }